# E-Commerce Sales & Operations Analytics

A practical walkthrough of Olist orders, item sales, customers, categories, delivery, reviews and payments. Item revenue is the sum of item prices and is not profit.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

DATA_DIR = Path('../data/raw')
orders = pd.read_csv(DATA_DIR / 'olist_orders_dataset.csv')
items = pd.read_csv(DATA_DIR / 'olist_order_items_dataset.csv')
customers = pd.read_csv(DATA_DIR / 'olist_customers_dataset.csv')
products = pd.read_csv(DATA_DIR / 'olist_products_dataset.csv')
sellers = pd.read_csv(DATA_DIR / 'olist_sellers_dataset.csv')
reviews = pd.read_csv(DATA_DIR / 'olist_order_reviews_dataset.csv')
payments = pd.read_csv(DATA_DIR / 'olist_order_payments_dataset.csv')
translation = pd.read_csv(DATA_DIR / 'product_category_name_translation.csv')
print('orders:', orders.shape, 'items:', items.shape, 'customers:', customers.shape)

## 1. Inspect structure and data quality

The source tables have different grains. Orders is one row per order; order items has multiple rows per order. Do not join item, payment and review detail together before aggregation.

In [ ]:
tables = {'orders': orders, 'items': items, 'customers': customers, 'products': products, 'sellers': sellers, 'reviews': reviews, 'payments': payments}
profile = pd.DataFrame([{'table': name, 'rows': len(df), 'columns': len(df.columns), 'duplicate_rows': df.duplicated().sum(), 'missing_cells': df.isna().sum().sum()} for name, df in tables.items()])
display(profile)
display(orders.isna().sum().sort_values(ascending=False).head(10))
print('Order key duplicates:', orders.order_id.duplicated().sum())
print('Order item key duplicates:', items.duplicated(['order_id', 'order_item_id']).sum())

## 2. Prepare dates and delivered sales

Delivered orders are used for realized sales and delivery analysis. Non-delivered orders remain relevant to order-status analysis. Freight is kept separate from item revenue.

In [ ]:
date_cols = ['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']
for col in date_cols:
    orders[col] = pd.to_datetime(orders[col], errors='coerce')
delivered = orders.loc[orders.order_status.eq('delivered')].copy()
sales = items.merge(delivered[['order_id', 'customer_id', 'order_purchase_timestamp', 'order_delivered_customer_date', 'order_estimated_delivery_date']], on='order_id', how='inner', validate='many_to_one')
sales['month'] = sales.order_purchase_timestamp.dt.to_period('M')
print('Delivered item revenue:', round(sales.price.sum(), 2))
print('Delivered orders:', sales.order_id.nunique())
print('Average order value:', round(sales.groupby('order_id').price.sum().mean(), 2))

## 3. Monthly sales

Monthly revenue is grouped by purchase month. The chart only includes delivered orders, so recent months may be incomplete if the extract ends before fulfillment catches up.

In [ ]:
monthly = sales.groupby('month').agg(revenue=('price', 'sum'), orders=('order_id', 'nunique')).reset_index()
fig, ax = plt.subplots(figsize=(11, 4))
sns.lineplot(data=monthly, x=monthly.month.astype(str), y='revenue', marker='o', ax=ax)
ax.set(title='Monthly delivered item revenue', xlabel='Purchase month', ylabel='Revenue'); ax.tick_params(axis='x', rotation=60); plt.tight_layout()
monthly.plot(x=monthly.month.astype(str), y='orders', kind='bar', figsize=(11, 4), legend=False, title='Delivered orders by purchase month'); plt.ylabel('Orders'); plt.tight_layout()

## 4. Category and geographic analysis

Translate category labels to English where available. Customer state is used for geographic sales because it is directly present in the customer table.

In [ ]:
category = sales.merge(products[['product_id', 'product_category_name']], on='product_id', how='left').merge(translation, on='product_category_name', how='left')
category['category'] = category.product_category_name_english.fillna(category.product_category_name).fillna('unknown')
category_sales = category.groupby('category').price.sum().sort_values(ascending=False)
category_sales.head(12).sort_values().plot.barh(figsize=(8, 5), title='Top categories by delivered item revenue'); plt.xlabel('Revenue'); plt.tight_layout()
state_sales = sales.merge(customers[['customer_id', 'customer_state']], on='customer_id').groupby('customer_state').price.sum().sort_values(ascending=False)
state_sales.head(10).sort_values().plot.barh(figsize=(7, 4), title='Top customer states by revenue'); plt.xlabel('Revenue'); plt.tight_layout()

## 5. Customer behavior

Use `customer_unique_id` as a person-level identifier. `customer_id` is linked to an individual order record in this dataset.

In [ ]:
customer_orders = delivered.merge(customers[['customer_id', 'customer_unique_id']], on='customer_id').groupby('customer_unique_id').order_id.nunique()
repeat_customers = int((customer_orders > 1).sum())
print('Unique delivered customers:', customer_orders.size)
print('Repeat customers:', repeat_customers)
print('Repeat share (%):', round((customer_orders > 1).mean() * 100, 2))

## 6. Reviews and delivery

Review text is optional; the numeric score supports satisfaction analysis. Compare at order level because an order can have more than one review record.

In [ ]:
reviews.review_creation_date = pd.to_datetime(reviews.review_creation_date, errors='coerce')
reviews.review_score.value_counts().sort_index().plot.bar(figsize=(6, 4), title='Review score distribution'); plt.xlabel('Score'); plt.ylabel('Review records'); plt.tight_layout()
delivery = delivered.dropna(subset=['order_delivered_customer_date']).copy()
delivery['delivery_days'] = (delivery.order_delivered_customer_date - delivery.order_purchase_timestamp).dt.total_seconds() / 86400
delivery['late'] = delivery.order_delivered_customer_date > delivery.order_estimated_delivery_date
sns.histplot(data=delivery, x='delivery_days', bins=40); plt.title('Purchase-to-customer delivery time'); plt.xlabel('Days'); plt.tight_layout()
print('Average delivery days:', round(delivery.delivery_days.mean(), 2))
print('Late share (%):', round(delivery.late.mean() * 100, 2))
order_score = reviews.groupby('order_id', as_index=False).review_score.mean()
review_delivery = delivery[['order_id', 'late']].merge(order_score, on='order_id')
print(review_delivery.groupby('late').review_score.mean())

## 7. Payments and validation

Payment value is transaction value, not product revenue. Keep it separate from item sales because installment rows and payment adjustments can make the measures differ.

In [ ]:
display(payments.groupby('payment_type').agg(payment_value=('payment_value', 'sum'), orders=('order_id', 'nunique')).sort_values('payment_value', ascending=False))
print('Item revenue:', round(sales.price.sum(), 2))
print('Freight on delivered items:', round(sales.freight_value.sum(), 2))
print('Payment value on delivered orders:', round(payments[payments.order_id.isin(delivered.order_id)].payment_value.sum(), 2))